In [18]:
import numpy as np
from PIL import Image
import random
import math
from typing import Tuple, List, Set, Optional, Union, Dict, Any

def load_image(image_path: str) -> Tuple[np.ndarray, Tuple[int, int, int]]:
    """Загрузка изображения и преобразование в numpy-массив."""
    img = Image.open(image_path)
    
    img_array: np.ndarray = np.array(img)
    points: np.ndarray = img_array.reshape(-1, img_array.shape[2])
    points = points.astype(np.float32) / 255.0
    
    return points, img_array.shape

def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Вычисляет евклидово расстояние между двумя точками."""
    return np.sqrt(np.sum((a - b) ** 2))

def knp_algorithm(points: np.ndarray, K: int) -> List[Tuple[int, int]]:
    """Алгоритм КНП."""
    n: int = len(points)
    if n <= 1:
        return []
    
    # Находим пару ближайших точек и объединяем их
    min_dist: float = float('inf')
    edge: Tuple[int, int] = (0, 1)
    for i in range(n):
        for j in range(i+1, n):
            dist: float = euclidean_distance(points[i], points[j])
            if dist < min_dist:
                min_dist = dist
                edge = (i, j)
    
    connected: Set[int] = set(edge)
    edges: List[Tuple[int, int]] = [edge]
    
    # Соединяем изолированные точки
    while len(connected) < n:
        min_dist = float('inf')
        new_edge: Optional[Tuple[int, int]] = None
        for i in range(n):
            if i not in connected:
                for j in connected:
                    dist = euclidean_distance(points[i], points[j])
                    if dist < min_dist:
                        min_dist = dist
                        new_edge = (i, j)
        if new_edge:
            edges.append(new_edge)
            connected.add(new_edge[0])
    
    # Удаляем K-1 самых длинных ребер
    if K > 1 and len(edges) > K-1:
        edges.sort(key=lambda e: -euclidean_distance(points[e[0]], points[e[1]]))
        edges = edges[-(len(edges)-(K-1)):]
    
    return edges

def forel_clustering(
    points: np.ndarray,
    R: float,
    K: int = 1,
    min_cluster_size: int = 1
) -> Tuple[List[List[int]], List[np.ndarray], List[Tuple[int, int]]]:
    """Реализация алгоритма кластеризации FOREL."""
    n_points: int = len(points)
    U: Set[int] = set(range(n_points))
    clusters: List[List[int]] = []
    centers: List[np.ndarray] = []
    
    while U:
        # Выбираем случайную начальную точку
        x0_idx: int = random.choice(list(U))
        x0: np.ndarray = points[x0_idx]
        
        while True:
            # Формируем кластер
            K0: List[int] = []
            for idx in U:
                if euclidean_distance(points[idx], x0) <= R:
                    K0.append(idx)
            
            if not K0:
                break
                
            new_x0: np.ndarray = np.mean(points[K0], axis=0)
            if euclidean_distance(new_x0, x0) < 1e-5:
                break
            x0 = new_x0
        
        if len(K0) < min_cluster_size:
            U -= set(K0)
            continue
            
        clusters.append(K0)
        centers.append(x0)
        U -= set(K0)
    
    # Применяем алгоритм КНП к центрам кластеров
    if len(centers) > 1:
        knp_edges: List[Tuple[int, int]] = knp_algorithm(np.array(centers), K)
    else:
        knp_edges = []
    
    # Приписываем точки к ближайшим центрам
    final_clusters: List[List[int]] = [[] for _ in range(len(centers))]
    for i in range(n_points):
        distances: List[float] = [euclidean_distance(points[i], c) for c in centers]
        closest: int = np.argmin(distances)
        final_clusters[closest].append(i)
    
    return final_clusters, centers, knp_edges

def visualize_clusters(
    clusters: List[List[int]],
    centers: List[np.ndarray],
    original_shape: Tuple[int, int, int]
) -> np.ndarray:
    """Визуализация кластеров: каждому кластеру присваивается средний цвет."""
    height, width = original_shape[0], original_shape[1]
    clustered: np.ndarray = np.zeros((height, width, original_shape[2]), dtype=np.uint8)
    
    for cluster_idx, cluster in enumerate(clusters):
        color: np.ndarray = (centers[cluster_idx] * 255).astype(np.uint8)
        for point_idx in cluster:
            y: int = point_idx // width
            x: int = point_idx % width
            clustered[y, x] = color
    
    return clustered

In [20]:
# Параметры
input_path = "origins/sk.jpg" 
output_path = "results/task_3/sk_clustered.jpg"

R = 0.3
K = 3

# Загрузка изображения
points, img_shape = load_image(input_path)

# Выполняем кластеризацию с помощью FOREL
clusters, centers, _ = forel_clustering(points, R, K)

# Визуализация
clustered_image = visualize_clusters(clusters, centers, img_shape)

# Сохраняем результат
result_img = Image.fromarray(clustered_image)
result_img.save(output_path)
result_img.show()
